# 日次ランキング（4指標）

旧07 + 旧14後半の統合版。日付ハードコードを廃止し、フォルダ内の最新日付を自動検出する。

| # | 指標 | 見えるもの |
|---|---|---|
| 1 | 単純差分 | 昨日→今日で最も伸びた動画（Shorts/長尺別） |
| 2 | 急増スコア（平均ベース） | 履歴平均と比べて急に伸びた動画 |
| 3 | 急増スコア（中央値ベース） | 外れ値に強い急増検知 |
| 4 | 日次増加の変化（3日差分） | 伸びが加速/減速している動画 |

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
drive.mount("/content/drive")

In [ ]:
#@title ⚙️ 設定と直近データのロード（最新2日を自動検出）
TOP_N = 10  #@param {type:"integer"}

from sixfonia_analytics import auth, config, load

youtube = auth.build_youtube()
days2 = load.load_latest_days(n_days=2)
date_prev, date_latest = sorted(days2.keys())

In [ ]:
#@title 1️⃣ 単純差分ランキング（Shorts / 長尺別）
from sixfonia_analytics import enrich, metrics, display as disp

simple_diffs = {}
for name in config.CHANNEL_NAMES:
    if name not in days2[date_latest] or name not in days2[date_prev]:
        print(f"[WARN] {name}: データ不足のためスキップ")
        continue
    merged = metrics.simple_daily_diff(days2[date_latest][name], days2[date_prev][name], name)
    # タイトル・投稿日・動画長（Shorts判定）を付与
    merged = enrich.add_video_details(merged.nlargest(TOP_N * 4, "viewCount_difference"), youtube)
    simple_diffs[name] = merged

for name, df in simple_diffs.items():
    disp.display_ranking_table(df[df["video_type"] == "Shorts"],
                               f"🎬 {name} — Shorts ランキング ({date_prev}→{date_latest})", top_n=TOP_N)
    disp.display_ranking_table(df[df["video_type"] == "Long-form"],
                               f"📹 {name} — 長尺 ランキング ({date_prev}→{date_latest})", top_n=TOP_N)

In [ ]:
#@title 2️⃣3️⃣ 急増スコア（平均ベース / 中央値ベース）
from sixfonia_analytics import load, metrics, display as disp

for name in simple_diffs:
    history = load.build_combined_df(name)
    base_diff = metrics.simple_daily_diff(days2[date_latest][name], days2[date_prev][name], name)

    for method, label in [("mean", "平均ベース"), ("median", "中央値ベース")]:
        scored = metrics.sudden_increase_scores(base_diff, history, method=method)
        top = scored.nlargest(TOP_N, "sudden_increase_score")
        top = enrich.add_video_details(top, youtube)
        disp.display_ranking_table(
            top, f"🚀 {name} — 急増スコア（{label}） Top {TOP_N}",
            metric_col="viewCount_difference", top_n=TOP_N,
        )
        display(top[["videoId", "video_title", "viewCount_difference",
                     f"{method}_daily_increase", "sudden_increase_score"]]
                .reset_index(drop=True))

In [ ]:
#@title 4️⃣ 日次増加の変化（3日差分: 伸びの加速/減速）
import pandas as pd
from sixfonia_analytics import load, metrics, display as disp

days3 = load.load_latest_days(n_days=3)
d2, d1, d0 = sorted(days3.keys())  # 古い→新しい

frames = []
for name in config.CHANNEL_NAMES:
    if not all(name in days3[d] for d in (d0, d1, d2)):
        continue
    frames.append(metrics.daily_increase_change(
        days3[d0][name], days3[d1][name], days3[d2][name], name))

change_df = pd.concat(frames, ignore_index=True)
top_change = change_df.nlargest(TOP_N * 2, "daily_increase_change")
top_change = enrich.add_video_details(top_change, youtube)

disp.display_ranking_table(
    top_change, f"⚡ 日次増加の変化 Top（全チャンネル / {d1}→{d0} vs {d2}→{d1}）",
    metric_col="daily_increase_change", top_n=TOP_N * 2,
)